In [2]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/Trabalho IA'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install transformers[tf] tensorflow

  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl (572.6 MB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.21.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.21.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.21.0 which is incompatible.


In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ViTModel, ViTImageProcessor

# Configuração de dispositivo (GPU se disponível, senão CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

# ==========================================
# 1. Configurações de Entrada e DataLoaders
# ==========================================
DATA_DIR = os.path.join(BASE_PATH, "letras_extraidas_transformer")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# No PyTorch, a normalização padrão do ViT (média e desvio padrão do ImageNet)
# substitui o escalonamento manual [-1, 1] do Keras.
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    # Normalização padrão que o ViT da Hugging Face espera:
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Carrega o dataset completo
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
num_classes = len(full_dataset.classes)

# Divisão de Treino e Validação (80% / 20%) equivalente ao validation_split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

# seed manual para reprodutibilidade
torch.manual_seed(123)
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Criação dos DataLoaders (equivalente ao image_dataset_from_directory)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 2. Construção do Modelo Vision Transformer
# ==========================================
class ViTClassifier(nn.Module):
    def __init__(self, num_classes):
        super(ViTClassifier, self).__init__()
        # Carrega o ViT pré-treinado em PyTorch (Sem o "TF")
        self.vit_base = ViTModel.from_pretrained('google/vit-base-patch16-224')

        # Congela o Transformer core inicialmente (Equivalente ao trainable = False)
        for param in self.vit_base.parameters():
            param.requires_grad = False

        # Cabeça de classificação customizada
        # O ViT-base tem uma dimensão oculta (hidden_size) de 768
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes) # Sem Softmax aqui, pois o CrossEntropyLoss já aplica internamente
        )

    def forward(self, pixel_values):
        # Passa pelo bloco do Transformer
        outputs = self.vit_base(pixel_values=pixel_values)

        # No PyTorch, o Hugging Face expõe o Class Token diretamente em 'pooler_output'
        # ou você pode pegar outputs.last_hidden_state[:, 0, :]
        cls_token = outputs.pooler_output

        # Passa pela cabeça customizada
        logits = self.classifier(cls_token)
        return logits

model = ViTClassifier(num_classes=num_classes).to(device)

# ==========================================
# 3. Otimizador, Função de Perda e Loop de Treino
# ==========================================
# Filtra para otimizar apenas os parâmetros que não estão congelados (a cabeça)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Equivalente ao sparse_categorical_crossentropy
criterion = nn.CrossEntropyLoss()

epochs = 10

print("Iniciando o treinamento (Warm-up da cabeça de classificação)...")
for epoch in range(epochs):
    # --- FASE DE TREINO ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Zera os gradientes
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass e Otimização
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = correct_train / total_train

    # --- FASE DE VALIDAÇÃO ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad(): # Desativa o cálculo de gradientes para economizar memória
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = correct_val / total_val

    print(f"Época [{epoch+1}/{epochs}] -> "
          f"Loss Treino: {epoch_train_loss:.4f} | Acc Treino: {epoch_train_acc:.4f} | "
          f"Loss Val: {epoch_val_loss:.4f} | Acc Val: {epoch_val_acc:.4f}")

Usando o dispositivo: cpu


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Iniciando o treinamento (Warm-up da cabeça de classificação)...


KeyboardInterrupt: 